### Generate CSV Files - Data from 1000 most recent matches
In order to do this, we will be using the [deadlock API](https://api.deadlock-api.com/docs#tag/matches/GET/v1/matches/metadata), which allows us to download a JSON file containing data from the most recent matches and their players. 1000 matches was selected arbitrarily. 

In order to generate the CSVs, we must first request the JSON data from the API, then filter it to just the data we want to analyze. Once this is done, we can input this data into a Pandas dataframe, then export it as two CSV files: one for individual player stats, and one for match stats. 

### Library Imports

In [1]:
import requests # to make an HTTP request to the API
import json # to process the HTTP request to a usable Python object
import pandas as pd # to format and export the data
from dataclasses import dataclass # technically optional, but it helps with organizing data

### API Options

In [2]:
# Can be set to "normal" or "streetbrawl"
# street brawl is an alternate, must quicker gamemode. 
# Though fun, it's a lot more randomness dependant, and the analysis isn't as interesting.
GAME_MODE = "normal"

# Can be set to "ranked" or "unranked"
# Ranked matches are more "serious" than unranked matches, as winning or losing will affect a player's rank. 
# players start at the "initiate" level rank, and can work their way up to eternus, the highest level available with all the top players. 
# I set it to ranked here, since players generally take ranked matches more seriously
MATCH_MODE = "ranked"

# Can be set to "start_time", "match_id", or "average_badge"
# start_time and match_id are vaguely interchangable, average badge sorts by players' ranks
ORDER = "start_time"
ORDER_DIRECTION = "desc"

# can be set to any integer
# sets the amount of matches to return
LIMIT = 1000

API_URL = f"https://api.deadlock-api.com/v1/matches/metadata?include_info=true&include_more_info=true&include_player_info=true&include_player_final_stats=true&game_mode={GAME_MODE}&match_mode={MATCH_MODE}&order_by={ORDER}&order_direction={ORDER_DIRECTION}&limit={LIMIT}&format=json"

### Retrieve and Process JSON

In [3]:
# using the URL we just generated, we first retrieve the raw JSON string
rawString: str = requests.get(API_URL)

# we then load it into a python dictionary
unformattedDict: dict = json.loads(rawString.text)

### Match Stats CSV

In [4]:
# This one will be a lot simpler than the player stats, since we're only tracking one actual statistic: the match length
# first, we'll organize everything into a dictionary
matchDict = {
    "matchID" : [x['match_id'] for x in unformattedDict],
    "startTime" : [x['start_time'] for x in unformattedDict],
    "duration" : [x['duration_s'] for x in unformattedDict]
}

# then, we create a dataframe with that dictionary and export it as a CSV
df = pd.DataFrame(data=matchDict)
df.to_csv("match_lengths.csv")

### Player Stats CSV
<b>NOTE:</b> heroesFiltered.json is required for this part to function, please run format_heroes.py before attempting to run this section

In [6]:
# in this file, we want to track each player's character, net worth, kills / deaths / assists, and whether their team won or not
# we'll eventually compute their KD Ratio (kills divided by deaths) using these statistics
# we'll also perform the processing necessary to display their character's actual name instead of the character's numerical ID

# first, since there's a lot to keep track of, we'll create a dataclass for each player's stats we want to keep track of
@dataclass
class PlayerStats:
    character: str
    netWorth: int
    won: bool
    kills: int
    deaths: int
    assists: int

# then, we initialize a dictionary that will map a character's numerical ID to their name
# this depends on the heroesFiltered.json file, see format_heroes.py for more info
with open("heroesFiltered.json", "r") as f:
    heroes = json.load(f)
    heroMap = {}
    for h in heroes:
        heroMap[h["id"]] = h["name"]

# we'll also make a small, arguably unnecessary helper function for determining if a player won or not
def playerWon(match, player):
    return player["team"] == match["winning_team"]

# now, for each match, we can loop through that match's players and add them to a list
statList = []
for match in unformattedDict:
    for player in match["players"]:
        stats = PlayerStats(
            heroMap[player["hero_id"]],
            player["net_worth"],
            playerWon(match, player),
            player["kills"],
            player["deaths"],
            player["assists"]
        )
        statList.append(stats)

# finally, convert statist to a dataframe and export it
statFrame = pd.DataFrame(data=statList)
statFrame.to_csv("player_stats.csv")